# UNIVERSAL ADAPTIVE PIPELINE: TEXT FEATURE ENGINEERING FOR TABULAR DATA
### COMPETITION-GRADE ARCHITECTURE (HYBRID MULTI-MODAL DATA MINING)

**Prinsip Desain:**
1. **Auto-Detection of Text Columns in Tabular Data**: Otomatis mendeteksi kolom mana saja di tabel yang bertipe teks panjang/deskripsi (misal: alasan pinjaman, deskripsi produk, riwayat diagnosa).
2. **Dense Feature Primitives**: Mengekstrak 3 set fitur numerik dari setiap kolom teks:
   - *Primitive A*: Morfometri struktural (panjang, rasio kapital, tanda baca, densitas angka).
   - *Primitive B*: Komponen semantik LSA (*TruncatedSVD*) yang memadatkan ribuan kata ke dalam 5–20 fitur angka kontinu.
   - *Primitive C*: Nilai keparahan/leksikal berbasis frekuensi.
3. **Seamless Feature Stitching**: Menggabungkan fitur teks yang telah diekstrak ke dalam tabel tabular numerik utama.
4. **LightGBM / CatBoost Training with Feature Importance**: Mengukur peningkatan performa model (*gain*) dari fitur teks yang baru dibuat.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, mean_squared_error
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
print("Environment Hybrid Feature Engineering siap. Seed:", SEED)


## 1. Unified Configuration Card


In [ ]:
CONFIG = {
    # Jalur dataset tabular yang memuat kolom teks
    "train_path": "dataset_hybrid_train.csv",
    "test_path": None,
    
    # Kolom: Set None untuk auto-detect
    "id_col": None,
    "target_col": None,
    "text_cols": None, # Jika None, mendeteksi semua kolom teks yang ada di tabel
    
    # Jumlah komponen topik LSA yang diekstrak per kolom teks
    "lsa_components_per_text_col": 10
}

print("Konfigurasi tersimpan.")


## 2. Auto-Discovery & Data Ingestion Engine


In [ ]:
def load_hybrid_data(cfg):
    tr_p = cfg["train_path"]
    if not os.path.exists(tr_p):
        print(f"Peringatan: Berkas '{tr_p}' belum ada. Menjalankan fallback generator tabular hibrida...")
        df = pd.DataFrame({
            "nasabah_id": range(1, 11),
            "usia": [25, 42, 33, 50, 29, 38, 45, 23, 60, 31],
            "pendapatan_juta": [8.5, 25.0, 12.0, 4.5, 9.0, 18.0, 6.0, 5.0, 30.0, 11.5],
            "jumlah_pinjaman_juta": [20.0, 100.0, 40.0, 15.0, 25.0, 70.0, 30.0, 10.0, 150.0, 35.0],
            "deskripsi_alasan_pinjaman": [
                "Modal tambahan untuk ekspansi usaha kedai kopi dan beli mesin espresso baru.",
                "Renovasi atap rumah yang bocor parah sebelum musim hujan datang.",
                "Biaya persalinan anak pertama di rumah sakit swasta dan kebutuhan bayi.",
                "Butuh dana darurat untuk melunasi utang pinjol yang sudah jatuh tempo bunga tinggi.",
                "Beli laptop baru untuk menunjang pekerjaan freelance desain grafis profesional.",
                "Investasi pembelian tanah kavling produktif di daerah pinggiran kota.",
                "Biaya pengobatan darurat orang tua yang sedang dirawat intensif di rumah sakit.",
                "Kebutuhan konsumtif beli motor baru untuk operasional harian kerja ojek online.",
                "Ekspansi bisnis restoran keluarga membuka cabang baru di pusat perbelanjaan.",
                "Modal perbaikan mobil operasional toko kelontong yang turun mesin rusak parah."
            ],
            "is_default": [0, 0, 0, 1, 0, 0, 1, 0, 0, 0] # Target biner: 1 = Gagal Bayar
        })
        test_df = None
    else:
        ext = os.path.splitext(tr_p)[1].lower()
        if ext in ['.xlsx', '.xls']: df = pd.read_excel(tr_p)
        elif ext == '.parquet': df = pd.read_parquet(tr_p)
        else: df = pd.read_csv(tr_p)
        test_df = None

    # Auto ID
    id_col = cfg["id_col"]
    if not id_col or id_col not in df.columns:
        id_cands = [c for c in df.columns if re.search(r"id|uuid|kode|no", c, re.IGNORECASE)]
        id_col = id_cands[0] if id_cands else df.columns[0]

    # Auto Target
    target_col = cfg["target_col"]
    if not target_col or target_col not in df.columns:
        target_cands = [c for c in df.columns if c != id_col and (df[c].nunique() == 2 or re.search(r"target|label|status|default", c, re.IGNORECASE))]
        target_col = target_cands[0] if target_cands else df.columns[-1]

    # Auto Text Columns: Deteksi semua kolom object/string yang rata-rata panjang karakternya > 20
    text_cols = cfg["text_cols"]
    if not text_cols:
        text_cols = [c for c in df.columns if c not in [id_col, target_col] and (df[c].dtype == object or pd.api.types.is_string_dtype(df[c])) and df[c].astype(str).str.len().mean() > 15]
        if not text_cols:
            text_cols = [c for c in df.columns if c not in [id_col, target_col] and df[c].dtype == object]
    elif isinstance(text_cols, str):
        text_cols = [text_cols]

    return df, test_df, id_col, target_col, text_cols

df, test_df, ID_COL, TARGET_COL, TEXT_COLS = load_hybrid_data(CONFIG)
print(f"=== HASIL DETEKSI SKEMA DATASET TABULAR HIBRIDA ===")
print(f"Kolom ID        : '{ID_COL}'")
print(f"Kolom Target    : '{TARGET_COL}'")
print(f"Kolom Teks Ditemukan : {TEXT_COLS}")
print(f"Total Kolom Asli: {df.shape[1]}")
print(df.head(2))


## 3. Ekstraksi Fitur Teks Otomatis (Morfometri + LSA Topik)


In [ ]:
new_text_features = []

for col in TEXT_COLS:
    print(f"Mengekstrak fitur dari kolom teks: '{col}'...")
    s = df[col].fillna("").astype(str)
    prefix = re.sub(r"[^\w]", "", col)[:10]
    
    # 1. Morfometri Struktural
    morf_df = pd.DataFrame(index=df.index)
    morf_df[f"{prefix}_char_len"] = s.str.len()
    morf_df[f"{prefix}_word_count"] = s.apply(lambda x: len(x.split()))
    morf_df[f"{prefix}_avg_word_len"] = morf_df[f"{prefix}_char_len"] / (morf_df[f"{prefix}_word_count"] + 1)
    morf_df[f"{prefix}_caps_ratio"] = s.apply(lambda x: sum(1 for c in x if c.isupper())) / (morf_df[f"{prefix}_char_len"] + 1)
    morf_df[f"{prefix}_digit_ratio"] = s.apply(lambda x: sum(1 for c in x if c.isdigit())) / (morf_df[f"{prefix}_char_len"] + 1)
    morf_df[f"{prefix}_punct_count"] = s.apply(lambda x: len(re.findall(r"[^\w\s]", x)))
    new_text_features.append(morf_df)
    
    # 2. LSA Semantic Components (Sublinear TF-IDF + TruncatedSVD)
    tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=1000, sublinear_tf=True)
    X_tfidf = tfidf.fit_transform(s.str.lower())
    
    n_comp = min(CONFIG["lsa_components_per_text_col"], X_tfidf.shape[1] - 1, len(df) - 1)
    n_comp = max(2, n_comp)
    svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
    X_lsa = svd.fit_transform(X_tfidf)
    
    lsa_df = pd.DataFrame(
        X_lsa, 
        columns=[f"{prefix}_lsa_dim_{i+1}" for i in range(n_comp)],
        index=df.index
    )
    new_text_features.append(lsa_df)

# Penggabungan seluruh fitur teks baru
extracted_features_df = pd.concat(new_text_features, axis=1)
print(f"Berhasil mengekstrak {extracted_features_df.shape[1]} fitur angka baru dari kolom teks.")


## 4. Feature Stitching: Penggabungan ke Matriks Tabular Utama


In [ ]:
# Hapus kolom teks asli dan gabungkan dengan fitur angka yang baru dibuat
df_enriched = pd.concat([
    df.drop(columns=TEXT_COLS),
    extracted_features_df
], axis=1)

print(f"Dimensi Awal    : {df.shape}")
print(f"Dimensi Diperkaya: {df_enriched.shape}")
print(f"Daftar Kolom Baru:\n{list(extracted_features_df.columns)}")


## 5. Pelatihan Model LightGBM & Analisis Feature Importance


In [ ]:
feature_cols = [c for c in df_enriched.columns if c not in [ID_COL, TARGET_COL]]
X = df_enriched[feature_cols]
y = df_enriched[TARGET_COL]

is_binary = df[TARGET_COL].nunique() == 2

params = {
    "objective": "binary" if is_binary else "regression",
    "metric": "auc" if is_binary else "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 8,
    "min_child_samples": 2,
    "random_state": SEED,
    "verbose": -1
}

train_data = lgb.Dataset(X, label=y)
model = lgb.train(params, train_data, num_boost_round=50)

# Analisis Kontribusi Fitur Teks terhadap Model
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importance(importance_type="gain")
}).sort_values(by="Importance", ascending=False)

print("=== 10 FITUR PALING BERPENGARUH TERHADAP PREDIKSI ===")
print(importance_df.head(10))
